# Building a Custom RAG Pipeline with LlamaIndex and Llama 3.2

This notebook builds a classic **vector RAG** pipeline with
[LlamaIndex](https://www.llamaindex.ai/): parse a PDF, embed it with a Hugging Face model,
index the embeddings, and answer questions with Groq's Llama 3.2 LLM.

See the accompanying `README.md` for setup and prerequisites.

---

*Written and developed by **Amin Amiri**. Released under the MIT License.*


## Import Libraries


In [ ]:
import os
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")    

## Data Ingestion

In [ ]:
import nest_asyncio

# LlamaParse is async-first, so we need to run this line because we are working on a notebook
nest_asyncio.apply()

In [ ]:
from llama_parse import LlamaParse
documents = LlamaParse(result_type="markdown").load_data("./data/2025_Tucson_Hybrid_user_manual.pdf")

# Setting up VectorStoreIndex

## LLM and Embedding Model

In [ ]:
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
llm = Groq(model="llama-3.2-3b-preview", api_key=GROQ_API_KEY)

Settings.llm = llm
Settings.embed_model = embed_model

## Initialize Vector Store Index

In [ ]:
vector_index = VectorStoreIndex.from_documents(documents, show_progress=True)

## RAG put into practice: Query engine

In [ ]:
query_engine = vector_index.as_query_engine()

res = query_engine.query("under what circumstances the navigation-based smart cruise control may not operate properly for the 2025 Tucson Hybrid?")

In [ ]:
from IPython.display import Markdown, display

display(Markdown(res.response))

# Persist

In [ ]:
PERSIST_DIR = "./storage"

if not os.path.exists(PERSIST_DIR):
    documents = LlamaParse(result_type="markdown").load_data("./data/2025_Tucson_Hybrid_user_manual.pdf")
    vector_index = VectorStoreIndex.from_documents(documents)

    # Store Vector Index
    vector_index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing Vector Index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    vector_index = load_index_from_storage(storage_context)

query_engine = vector_index.as_query_engine()
res = query_engine.query("under what circumstances the navigation-based smart cruise control may not operate properly for the 2025 Tucson Hybrid?")

display(Markdown(res.response))